In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [2]:
model_name = "uer/roberta-base-finetuned-jd-binary-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [3]:
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [4]:
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((76

In [5]:
texts = [
    "这个手机拍照效果非常好，很满意！",
    "质量太差了，用了一天就坏了，差评。",
    "一般般吧，没什么特别的。",
]

In [6]:
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [7]:
inputs

{'input_ids': tensor([[ 101, 6821,  702, 2797, 3322, 2864, 4212, 3126, 3362, 7478, 2382, 1962,
         8024, 2523, 4007, 2692, 8013,  102,    0],
        [ 101, 6574, 7030, 1922, 2345,  749, 8024, 4500,  749,  671, 1921, 2218,
         1776,  749, 8024, 2345, 6397,  511,  102],
        [ 101,  671, 5663, 5663, 1416, 8024, 3766,  784,  720, 4294, 1166, 4638,
          511,  102,    0,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])}

In [8]:
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

In [9]:
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-2.5172,  2.5376],
        [ 2.5171, -2.6075],
        [-0.1967,  0.1181]]), hidden_states=None, attentions=None)

In [10]:
logits

tensor([[-2.5172,  2.5376],
        [ 2.5171, -2.6075],
        [-0.1967,  0.1181]])

In [11]:
preditions = torch.argmax(logits, dim=-1)
labels = ['负面', '正面']

In [12]:
for text, pred in zip(texts, preditions):
    print(f"{text} → {labels[pred.item()]}")

这个手机拍照效果非常好，很满意！ → 正面
质量太差了，用了一天就坏了，差评。 → 负面
一般般吧，没什么特别的。 → 正面


In [14]:
probs = torch.softmax(logits, dim=-1)

for text, prob in zip(texts, probs):
    neg, pos = prob.tolist()
    print(f"{text}")
    print(f"  负面: {neg:.4f}  正面: {pos:.4f}\n")

这个手机拍照效果非常好，很满意！
  负面: 0.0063  正面: 0.9937

质量太差了，用了一天就坏了，差评。
  负面: 0.9941  正面: 0.0059

一般般吧，没什么特别的。
  负面: 0.4219  正面: 0.5781



In [15]:
from transformers import pipeline

In [16]:
classifier = pipeline("sentiment-analysis", model=model_name)

for text in texts:
    result = classifier(text)
    print(f"{text} → {result[0]['label']} ({result[0]['score']:.4f})")

Device set to use cuda:0


这个手机拍照效果非常好，很满意！ → positive (stars 4 and 5) (0.9937)
质量太差了，用了一天就坏了，差评。 → negative (stars 1, 2 and 3) (0.9941)
一般般吧，没什么特别的。 → positive (stars 4 and 5) (0.5781)
